# Birds v3 — CNN (PyTorch)

**Why CNN over v2's LightGBM ensemble (43.4%)?**  
v1/v2 flattened the 64×32 spectrogram to a 1D vector, discarding spatial structure. A CNN keeps the 2D layout, learns local time-frequency patterns via shared filters, and builds hierarchical features — the standard approach for any image task.

**Architecture**: 4 convolutional blocks (Conv→BN→GELU×2 → MaxPool → Dropout2d) + Global Average Pooling + FC head.  
**Augmentation**: SpecAugment-style time/frequency masking + Gaussian noise.  
**Scheduler**: Linear warmup + cosine annealing.  
**Loss**: Label-smoothing cross-entropy (smoothing=0.1).

**AI note**: Claude Code (claude-sonnet-4-6) wrote this after reading v1/v2 results and checking the py311 environment. All modelling decisions reviewed by the student.

In [ ]:
import os, time, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')

BASE      = Path('All_about_Birds/All_about_Birds')
TRAIN_DIR = BASE / 'train'
TEST_DIR  = BASE / 'test'
SAMPLE    = Path('sample_submission.csv')

IMG_H, IMG_W = 32, 64          # height × width (frequency × time)
N_CLASSES    = 9
SEED         = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('Train classes:', [d.name for d in sorted(TRAIN_DIR.iterdir()) if d.is_dir()])

## 1. Dataset

In [ ]:
class BirdDataset(Dataset):
    """Loads 64×32 grayscale PNGs. Augmentation uses SpecAugment-style masking."""

    def __init__(self, paths, labels=None, augment=False):
        self.paths   = paths
        self.labels  = labels
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.array(
            Image.open(self.paths[idx]).convert('L'), dtype=np.float32
        ) / 255.0  # (H, W) in [0,1]

        if self.augment:
            img = self._augment(img)

        # Add channel dim → (1, H, W)
        tensor = torch.from_numpy(img[np.newaxis])

        if self.labels is not None:
            return tensor, int(self.labels[idx])
        return tensor

    def _augment(self, img):
        mean_val = float(img.mean())

        # Time masking (horizontal strips)
        if random.random() < 0.5:
            t = random.randint(0, IMG_W - 1)
            w = random.randint(1, max(1, IMG_W // 4))
            img[:, t:t+w] = mean_val

        # Frequency masking (vertical strips)
        if random.random() < 0.5:
            f = random.randint(0, IMG_H - 1)
            h = random.randint(1, max(1, IMG_H // 4))
            img[f:f+h, :] = mean_val

        # Small Gaussian noise
        if random.random() < 0.3:
            img = img + np.random.randn(*img.shape).astype(np.float32) * 0.02
            img = np.clip(img, 0.0, 1.0)

        return img

print('Dataset class defined.')

## 2. CNN Architecture

Input `(B, 1, 32, 64)` → four conv blocks with progressive doubling of channels → Global Average Pool → FC head.

| Layer | Output shape |
|---|---|
| Input | (B, 1, 32, 64) |
| Block 1 (32 ch) + MaxPool | (B, 32, 16, 32) |
| Block 2 (64 ch) + MaxPool | (B, 64, 8, 16) |
| Block 3 (128 ch) + MaxPool | (B, 128, 4, 8) |
| Block 4 (256 ch) + MaxPool | (B, 256, 2, 4) |
| Global Avg Pool | (B, 256) |
| FC head | (B, 9) |

In [ ]:
def conv_block(in_ch, out_ch, dropout=0.1):
    return nn.Sequential(
        nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch), nn.GELU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch), nn.GELU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(dropout),
    )

class BirdCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1,   32,  dropout=0.05),
            conv_block(32,  64,  dropout=0.10),
            conv_block(64,  128, dropout=0.10),
            conv_block(128, 256, dropout=0.15),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

# Sanity check
_m = BirdCNN()
_x = torch.zeros(2, 1, IMG_H, IMG_W)
_y = _m(_x)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Output shape: {_y.shape}  |  Parameters: {n_params:,}')
del _m, _x, _y

## 3. Collect Paths & Split

In [ ]:
all_paths, all_labels = [], []
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    pngs = sorted(cls_dir.glob('*.png'))
    all_paths.extend(pngs)
    all_labels.extend([cls_dir.name] * len(pngs))
    print(f'  {cls_dir.name:35s} {len(pngs):5d}')

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

le = LabelEncoder()
y_enc = le.fit_transform(all_labels)
print(f'\nTotal: {len(all_paths)} images, {N_CLASSES} classes')
print('Classes:', list(le.classes_))

tr_paths, val_paths, y_tr, y_val = train_test_split(
    all_paths, y_enc, test_size=0.15, random_state=SEED, stratify=y_enc
)
print(f'Train: {len(tr_paths)}  Val: {len(val_paths)}')

# Test paths (in submission order)
sample_sub     = pd.read_csv(SAMPLE)
test_filenames = sample_sub['file_name'].tolist()
test_paths     = [TEST_DIR / f for f in test_filenames]

## 4. DataLoaders

In [ ]:
BATCH_SIZE  = 128
NUM_WORKERS = 2   # reduce to 0 if you get DataLoader errors on WSL

train_ds = BirdDataset(tr_paths,   y_tr,  augment=True)
val_ds   = BirdDataset(val_paths,  y_val, augment=False)
test_ds  = BirdDataset(test_paths,        augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

print(f'Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}')

## 5. Training

In [ ]:
EPOCHS    = 60
LR        = 3e-4
WARMUP    = 5        # epochs of linear warmup
SMOOTHING = 0.1

model     = BirdCNN().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)

# Linear warmup then cosine annealing
def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (logits.argmax(1) == labels).sum().item()
            n          += len(imgs)
    return total_loss / n, correct / n

best_val_acc  = 0.0
best_state    = None
history       = []

print(f'Training for {EPOCHS} epochs on {DEVICE}...')
print(f'{"Ep":>4}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}  {"LR":>8}  Time')
print('-' * 70)

t_total = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                    'va_loss': va_loss, 'va_acc': va_acc})

    marker = ' *' if va_acc > best_val_acc else ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'{epoch:4d}  {tr_loss:10.4f}  {tr_acc:9.4f}  {va_loss:8.4f}  '
          f'{va_acc:7.4f}  {current_lr:8.6f}  {elapsed:.0f}s{marker}')

print(f'\nDone in {(time.time()-t_total)/60:.1f} min  |  Best val acc: {best_val_acc:.4f}')

## 6. Per-class Report (Best Checkpoint)

In [ ]:
model.load_state_dict(best_state)
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

print(f'Val accuracy: {accuracy_score(all_true, all_preds):.4f}')
print(classification_report(all_true, all_preds, target_names=le.classes_))

## 7. Retrain on Full Training Set

We use the same hyperparameters but train on 100% of the labelled data (no val split).  
We stop at `EPOCHS` epochs (no early stopping available without a val set).

In [ ]:
print('Retraining on full labelled dataset...')
t0 = time.time()

full_ds     = BirdDataset(all_paths, y_enc, augment=True)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

model2     = BirdCNN().to(DEVICE)
optimizer2 = optim.AdamW(model2.parameters(), lr=LR, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.LambdaLR(optimizer2, lr_lambda)

model2.train()
for epoch in range(1, EPOCHS + 1):
    for imgs, labels in full_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer2.zero_grad()
        loss = criterion(model2(imgs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer2.step()
    scheduler2.step()
    if epoch % 10 == 0:
        print(f'  epoch {epoch}/{EPOCHS}  ({time.time()-t0:.0f}s elapsed)')

print(f'Done in {(time.time()-t0)/60:.1f} min')

## 8. Generate Submission

In [ ]:
model2.eval()
test_preds_enc = []

with torch.no_grad():
    for imgs in test_loader:
        # test_loader yields single tensor (no labels)
        preds = model2(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        test_preds_enc.extend(preds)

test_labels = le.inverse_transform(test_preds_enc)

submission = pd.DataFrame({'file_name': test_filenames, 'label': test_labels})
submission.to_csv('submission_v3_cnn.csv', index=False)
print(f'Saved submission_v3_cnn.csv ({len(submission)} rows)')
print('Distribution:')
print(pd.Series(test_labels).value_counts().sort_index())

## 9. (Optional) Test-Time Augmentation (TTA)

Average predictions over multiple augmented views of each test image — typically adds +1–3% accuracy.

In [ ]:
TTA_ROUNDS = 5

model2.eval()
tta_probs = None

softmax = nn.Softmax(dim=1)

for round_idx in range(TTA_ROUNDS):
    tta_ds     = BirdDataset(test_paths, augment=(round_idx > 0))  # first round: no aug
    tta_loader = DataLoader(tta_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS)
    round_probs = []
    with torch.no_grad():
        for imgs in tta_loader:
            probs = softmax(model2(imgs.to(DEVICE))).cpu().numpy()
            round_probs.append(probs)
    round_probs = np.concatenate(round_probs, axis=0)
    tta_probs   = round_probs if tta_probs is None else tta_probs + round_probs
    print(f'TTA round {round_idx+1}/{TTA_ROUNDS}')

tta_preds  = tta_probs.argmax(axis=1)
tta_labels = le.inverse_transform(tta_preds)

submission_tta = pd.DataFrame({'file_name': test_filenames, 'label': tta_labels})
submission_tta.to_csv('submission_v3_cnn_tta.csv', index=False)
print(f'\nSaved submission_v3_cnn_tta.csv ({len(submission_tta)} rows)')
print('Distribution:')
print(pd.Series(tta_labels).value_counts().sort_index())